## 1. Ringkasan file BS dalam Format csv

In [5]:
import pandas as pd

# Baca data asli dengan parsing kolom waktu sebagai datetime
df = pd.read_csv('dataset/data_all.csv', parse_dates=['start time', 'end time'])

# Hitung durasi koneksi dalam menit untuk setiap baris data
df['duration_minutes'] = (df['end time'] - df['start time']).dt.total_seconds() // 60

# Kelompokkan data berdasarkan kolom 'address'
grouped = df.groupby('address')

# Ambil satu baris representatif (pertama) dari setiap grup address
df_unique = grouped.first().reset_index()

# Tambahkan kolom total durasi koneksi per address
df_unique['duration_minutes'] = grouped['duration_minutes'].sum().values

# Tambahkan kolom jumlah kunjungan (jumlah baris data) per address
df_unique['visits'] = grouped.size().values

# Tambahkan kolom BSID sebagai ID unik untuk setiap base station
df_unique.insert(0, 'BSID', range(len(df_unique)))

# Simpan hasil data unik ke file CSV tanpa menyimpan index
df_unique.to_csv('dataset/bs_unique.csv', index=False)

print("Data unik base station telah disimpan ke 'dataset/bs_unique.csv'")

Data unik base station telah disimpan ke 'bs_unique.csv'


## 2. Menampilkan Peta Format html

In [15]:
import pandas as pd
import folium

# Baca data dari bs_unique.csv
df = pd.read_csv('dataset/bs_unique.csv')

# Validasi kolom yang dibutuhkan
required_cols = ['BSID', 'address', 'latitude', 'longitude', 'duration_minutes', 'visits']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan di bs_unique.csv: {missing_cols}")

# Hitung rata-rata koordinat untuk pusat peta
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

# Inisialisasi peta
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

# Tambahkan marker untuk setiap base station
for _, row in df.iterrows():
    popup_text = (
        f"<b>BSID:</b> {row['BSID']}<br>"
        f"<b>Address:</b> {row['address']}<br>"
        f"<b>Total Workload:</b> {row['duration_minutes']:.0f} menit<br>"
        f"<b>Total Visits:</b> {row['visits']}"
    )
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='blue',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Simpan ke HTML
output_file = 'results/html/bs_map.html'
m.save(output_file)
print(f"✅ Peta telah disimpan ke: {output_file}")

Peta telah disimpan ke 'bs_map.html'


## 3. Visualisasi BS dan Hasil Optimasi

In [19]:
import pandas as pd
import folium
import json
import os
from math import radians, cos, sin, asin, sqrt

# === Load Base Station Data (ID, koordinat, workload, visits) ===
df_bs = pd.read_csv('dataset/bs_unique.csv')

# === Folder lokasi input JSON dan output HTML ===
input_folder = 'results/json'
output_folder = 'results/html'

# === Fungsi Haversine untuk hitung jarak antar koordinat dalam km ===
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # radius bumi (km)
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# === Loop semua file lokasi server ===
for filename in os.listdir(input_folder):
    if filename.endswith('_locations.json'):
        json_path = os.path.join(input_folder, filename)
        output_name = filename.replace('_locations.json', '.html')
        output_path = os.path.join(output_folder, output_name)

        with open(json_path, 'r') as f:
            server_data = json.load(f)

        if not server_data:
            print(f"⚠️  Lewati (kosong): {filename}")
            continue

        # Pusatkan peta berdasarkan rata-rata lokasi server
        center_lat = sum([s['latitude'] for s in server_data]) / len(server_data)
        center_lon = sum([s['longitude'] for s in server_data]) / len(server_data)
        m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

        # Simpan relasi BS ke ES (untuk ditampilkan pada pop-up BS)
        bs_to_es_info = {}

        # === Plot semua Edge Server (marker merah) ===
        for i, server in enumerate(server_data):
            server_lat = server['latitude']
            server_lon = server['longitude']
            assigned_bs = server.get('assigned_bs', [])

            bs_count = len(assigned_bs)
            total_workload = 0
            total_visits = 0
            total_distance_km = 0.0

            for bs in assigned_bs:
                matched = df_bs[
                    (df_bs['latitude'].round(6) == round(bs['latitude'], 6)) &
                    (df_bs['longitude'].round(6) == round(bs['longitude'], 6))
                ]
                if not matched.empty:
                    total_workload += matched.iloc[0]['duration_minutes']
                    total_visits += matched.iloc[0]['visits']

                dist = haversine(server_lat, server_lon, bs['latitude'], bs['longitude'])
                total_distance_km += dist

                # Simpan informasi koneksi BS ke ES
                key = (round(bs['latitude'], 6), round(bs['longitude'], 6))
                bs_to_es_info[key] = {
                    'edge_id': i + 1,
                    'distance': dist
                }

            popup_text = (
                f"<b>Edge Server {i+1}</b><br>"
                f"<b>Jumlah BS:</b> {bs_count}<br>"
                f"<b>Total Workload:</b> {total_workload:.0f} menit<br>"
                f"<b>Total Visits:</b> {total_visits}<br>"
                f"<b>Total Distance:</b> {total_distance_km:.2f} km"
            )

            folium.Marker(
                location=[server_lat, server_lon],
                popup=folium.Popup(popup_text, max_width=300),
                tooltip=f"Edge Server {i+1}",
                icon=folium.Icon(color='red', icon='cloud', prefix='fa')
            ).add_to(m)

        # === Plot semua Base Station (marker biru kecil) ===
        for _, row in df_bs.iterrows():
            lat = round(row['latitude'], 6)
            lon = round(row['longitude'], 6)
            key = (lat, lon)

            # Cek apakah BS ini punya edge server yang terhubung
            es_info = bs_to_es_info.get(key, None)

            es_text = f"<i>Not connected to any edge server</i>"
            if es_info:
                es_text = (
                    f"Connected to ES #{es_info['edge_id']}<br>"
                    f"Distance to ES: {es_info['distance']:.2f} km"
                )

            popup_text = (
                f"<b>BSID:</b> {row['BSID']}<br>"
                f"<b>Address:</b> {row['address']}<br>"
                f"<b>Total Workload:</b> {row['duration_minutes']:.0f} menit<br>"
                f"<b>Total Visits:</b> {row['visits']}<br>"
                f"{es_text}"
            )

            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=3,
                color='blue',
                fill=True,
                fill_opacity=0.6,
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(m)

        m.save(output_path)
        print(f"✅ Peta disimpan: {output_path}")

print("✅ Seluruh Peta telah tersimpan")

✅ Peta dengan popup disimpan ke: results/html/kmeans_k100.html
